In [2]:
!pip install tensorflow-model-optimization

     ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 61.0/61.0 kB 2.3 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 242.5/242.5 kB 8.5 MB/s eta 0:00:00
   ━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━━ 18.0/18.0 MB 53.2 MB/s eta 0:00:00
  Attempting uninstall: numpy
    Found existing installation: numpy 2.0.2
    Uninstalling numpy-2.0.2:
      Successfully uninstalled numpy-2.0.2
ERROR: pip's dependency resolver does not currently take into account all the packages that are installed. This behaviour is the source of the following dependency conflicts.
tobler 0.14.0 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
rasterio 1.5.0 requires numpy>=2, but you have numpy 1.26.4 which is incompatible.
jax 0.7.2 requires numpy>=2.0, but you have numpy 1.26.4 which is incompatible.
opencv-python-headless 4.13.0.92 requires numpy>=2; python_version >= "3.9", but you have numpy 1.26.4 which is incompatible.
pytensor 2.38.2 requires numpy>=2.0, but you have numpy 1.26.4 wh

In [1]:
# ------------------------------------------------------------
# CIFAR-10 CNN: Baseline + Quantization (Pruning Removed)
# ------------------------------------------------------------

import tensorflow as tf
from tensorflow.keras import layers, models
from tensorflow.keras.datasets import cifar10
import numpy as np
import time
import os
import matplotlib.pyplot as plt

# ------------------------------------------------------------
# 1. Load CIFAR-10
# ------------------------------------------------------------
(x_train, y_train), (x_test, y_test) = cifar10.load_data()
x_train = x_train.astype("float32") / 255.0
x_test = x_test.astype("float32") / 255.0

y_train = tf.keras.utils.to_categorical(y_train, 10)
y_test = tf.keras.utils.to_categorical(y_test, 10)

# ------------------------------------------------------------
# 2. Build Baseline CNN
# ------------------------------------------------------------
def create_baseline_model():
    model = models.Sequential([
        layers.Input(shape=(32, 32, 3)),
        layers.Conv2D(32, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(64, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Conv2D(128, (3, 3), activation="relu", padding="same"),
        layers.MaxPooling2D((2, 2)),
        layers.Flatten(),
        layers.Dense(128, activation="relu"),
        layers.Dense(10, activation="softmax")
    ])
    model.compile(optimizer="adam",
                  loss="categorical_crossentropy",
                  metrics=["accuracy"])
    return model

baseline_model = create_baseline_model()

# ------------------------------------------------------------
# 3. Train Baseline Model
# ------------------------------------------------------------
history = baseline_model.fit(
    x_train, y_train,
    epochs=15,
    batch_size=64,
    validation_split=0.2
)

# ------------------------------------------------------------
# 4. Evaluate Baseline Model
# ------------------------------------------------------------
test_loss, test_acc = baseline_model.evaluate(x_test, y_test, verbose=0)
print("\nBaseline Test Accuracy:", test_acc)

baseline_model.save("baseline_model.h5")
size_mb = os.path.getsize("baseline_model.h5") / (1024 * 1024)
print("Baseline Model Size (MB):", size_mb)

# Latency
sample = x_test[:200]
start = time.time()
_ = baseline_model.predict(sample, verbose=0)
end = time.time()
latency_ms = (end - start) / len(sample) * 1000
print("Baseline Latency (ms/image):", latency_ms)

# ------------------------------------------------------------
# 5. Quantization (TFLite)
# ------------------------------------------------------------
converter = tf.lite.TFLiteConverter.from_keras_model(baseline_model)
converter.optimizations = [tf.lite.Optimize.DEFAULT]
quant_model = converter.convert()

with open("quantized_model.tflite", "wb") as f:
    f.write(quant_model)

q_size = os.path.getsize("quantized_model.tflite") / (1024 * 1024)
print("\nQuantized Model Size (MB):", q_size)

# Evaluate quantized model
interpreter = tf.lite.Interpreter(model_path="quantized_model.tflite")
interpreter.allocate_tensors()

input_details = interpreter.get_input_details()
output_details = interpreter.get_output_details()

def tflite_predict(images):
    preds = []
    for img in images:
        img = np.expand_dims(img, axis=0).astype(input_details[0]["dtype"])
        interpreter.set_tensor(input_details[0]["index"], img)
        interpreter.invoke()
        output = interpreter.get_tensor(output_details[0]["index"])
        preds.append(output)
    return np.vstack(preds)

x_eval = x_test[:500]
y_eval = y_test[:500]

y_pred = tflite_predict(x_eval)
q_acc = np.mean(np.argmax(y_pred, axis=1) == np.argmax(y_eval, axis=1))
print("Quantized Model Accuracy (subset):", q_acc)

# Latency
start = time.time()
_ = tflite_predict(sample)
end = time.time()
q_latency = (end - start) / len(sample) * 1000
print("Quantized Latency (ms/image):", q_latency)

# ------------------------------------------------------------
# 6. Summary
# ------------------------------------------------------------
print("\n---------------- SUMMARY ----------------")
print(f"Baseline Accuracy: {test_acc:.4f}, Size: {size_mb:.2f} MB, Latency: {latency_ms:.2f} ms")
print(f"Quantized Acc:     {q_acc:.4f}, Size: {q_size:.2f} MB, Latency: {q_latency:.2f} ms")
print("-----------------------------------------")

170498071/170498071 ━━━━━━━━━━━━━━━━━━━━ 6s 0us/step
Epoch 1/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 91s 143ms/step - accuracy: 0.4463 - loss: 1.5235 - val_accuracy: 0.5588 - val_loss: 1.2323
Epoch 2/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 89s 143ms/step - accuracy: 0.6141 - loss: 1.0966 - val_accuracy: 0.6413 - val_loss: 1.0262
Epoch 3/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 140s 139ms/step - accuracy: 0.6779 - loss: 0.9189 - val_accuracy: 0.6905 - val_loss: 0.8926
Epoch 4/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 142s 139ms/step - accuracy: 0.7192 - loss: 0.8072 - val_accuracy: 0.6900 - val_loss: 0.8927
Epoch 5/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 144s 142ms/step - accuracy: 0.7505 - loss: 0.7175 - val_accuracy: 0.7213 - val_loss: 0.8069
Epoch 6/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 146s 149ms/step - accuracy: 0.7823 - loss: 0.6296 - val_accuracy: 0.7354 - val_loss: 0.7903
Epoch 7/15
625/625 ━━━━━━━━━━━━━━━━━━━━ 90s 144ms/step - accuracy: 0.8057 - loss: 0.5555 - val_accuracy: 0.7361 - val_loss: 0.7921
Epoch 8/15
625/625 ━━━━━━━


Baseline Test Accuracy: 0.7188000082969666
Baseline Model Size (MB): 4.1274566650390625
Baseline Latency (ms/image): 1.6429650783538818
Saved artifact at '/tmp/tmpsklza_3d'. The following endpoints are available:

* Endpoint 'serve'
  args_0 (POSITIONAL_ONLY): TensorSpec(shape=(None, 32, 32, 3), dtype=tf.float32, name='keras_tensor')
Output Type:
  TensorSpec(shape=(None, 10), dtype=tf.float32, name=None)
Captures:
  137228268396880: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137228268399568: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137228268398224: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137228268395344: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137228268398992: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137228268398608: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137228268397264: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137228268397072: TensorSpec(shape=(), dtype=tf.resource, name=None)
  137228268398800: T

/usr/local/lib/python3.12/dist-packages/tensorflow/lite/python/interpreter.py:457: UserWarning:     Warning: tf.lite.Interpreter is deprecated and is scheduled for deletion in
    TF 2.20. Please use the LiteRT interpreter from the ai_edge_litert package.
    See the [migration guide](https://ai.google.dev/edge/litert/migration)
    for details.
    
  warnings.warn(_INTERPRETER_DELETION_WARNING)


Quantized Model Accuracy (subset): 0.708
Quantized Latency (ms/image): 0.42409181594848633

---------------- SUMMARY ----------------
Baseline Accuracy: 0.7188, Size: 4.13 MB, Latency: 1.64 ms
Quantized Acc:     0.7080, Size: 0.35 MB, Latency: 0.42 ms
-----------------------------------------
